# Practical 6 – Object Detection using SSD MobileNetV2

**Aim:** Load a pretrained SSD MobileNetV2 detector, run it on an image, and draw labeled bounding boxes for detections above a confidence threshold.

Every important code line is explained using `# comments` directly inside the code. Run the cells from top to bottom.

## 1. Install packages

In [ ]:
# Install the libraries used for deep learning, pretrained models, image processing, and plotting.
# Run this cell only when these packages are not already installed.
%pip install -q tensorflow tensorflow-hub opencv-python matplotlib

# Upgrade pip, the Python package installer.
%pip install --upgrade pip

# Install the setuptools version used in the original practical setup.
%pip install setuptools==80.9.0

## 2. Import libraries and load model

In [ ]:
# TensorFlow runs the object-detection model.
import tensorflow as tf

# TensorFlow Hub loads the pretrained SSD MobileNetV2 model.
import tensorflow_hub as hub

# OpenCV reads and annotates images.
import cv2

# Matplotlib displays the original and detected images.
import matplotlib.pyplot as plt

# Download/load the pretrained SSD MobileNetV2 object detector.
model = hub.load("https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2")

# Confirm that the model has loaded.
print("SSD MobileNetV2 model loaded successfully")

## 3. Read and display input image

In [ ]:
# Set the image filename. Keep testm.jpeg in the same folder as this notebook,
# or replace this value with the full path to your own image.
image_path = "testm.jpeg"

# Read the image from disk; OpenCV loads it in BGR channel order.
image = cv2.imread(image_path)

# Stop with a clear error if the file could not be read.
if image is None:
    raise FileNotFoundError("Image not found! Check image_path and file location.")

# Convert BGR to RGB so Matplotlib displays the colors correctly.
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Display the original image without axes.
plt.imshow(image)
plt.axis("off")
plt.show()

## 4. Prepare image tensor and run detection

In [ ]:
# Convert the RGB image array into a TensorFlow tensor of 8-bit pixel values.
input_image = tf.convert_to_tensor(image, dtype=tf.uint8)

# Add a batch dimension: one image becomes a batch containing one image.
input_image = tf.expand_dims(input_image, axis=0)

# Display the shape supplied to the model.
print("Input tensor shape:", input_image.shape)

# Run the detector and store its output dictionary.
result = model(input_image)

# Confirm inference has completed and display available output names.
print("Detection completed")
print("Available result keys:", list(result.keys()))

## 5. Extract detection results

In [ ]:
# Extract normalized bounding-box coordinates for the first image in the batch.
boxes = result["detection_boxes"][0].numpy()

# Extract the predicted class IDs for the first image.
classes = result["detection_classes"][0].numpy()

# Extract confidence scores for the first image.
scores = result["detection_scores"][0].numpy()

# Display how many candidate detections the model returned.
print("Number of detections returned:", len(scores))

# Read image height and width; ignore the channel count using underscore.
h, w, _ = image.shape

## 6. Map class IDs to names and draw boxes

In [ ]:
# Map selected COCO class IDs to readable labels.
labels = {
    1: "person",
    2: "bicycle",
    3: "car",
    4: "motorcycle",
    5: "airplane",
    6: "bus",
    7: "train",
    8: "truck",
    9: "boat",
    17: "cat",
    18: "dog",
    19: "horse",
    20: "sheep",
    21: "cow"
}

# Visit every detection returned by the model.
for i in range(len(scores)):

    # Keep only detections with confidence strictly greater than 0.5.
    if scores[i] > 0.5:

        # Each box is ordered as normalized [top, left, bottom, right].
        y1, x1, y2, x2 = boxes[i]

        # Convert normalized horizontal coordinates into image pixel positions.
        x1, x2 = int(x1 * w), int(x2 * w)

        # Convert normalized vertical coordinates into image pixel positions.
        y1, y2 = int(y1 * h), int(y2 * h)

        # Draw a green rectangle around the detected object; thickness is 2 pixels.
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Convert the predicted class value to an integer ID.
        class_id = int(classes[i])

        # Find the readable class name; use 'Object' if the ID is not in our dictionary.
        label = labels.get(class_id, "Object")

        # Draw the label just above the box; max(..., 0) keeps text within the image.
        cv2.putText(
            image, label, (x1, max(y1 - 10, 0)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2
        )

# Confirm that boxes were drawn for detections passing the confidence threshold.
print("Bounding boxes drawn for detections with confidence > 0.5")

## 7. Display final object-detection output

In [ ]:
# Create a larger figure for the final annotated image.
plt.figure(figsize=(10, 8))

# Display the image after bounding boxes and labels have been drawn.
plt.imshow(image)

# Hide coordinate axes.
plt.axis('off')

# Add a descriptive title.
plt.title('Object Detection using SSD MobileNetV2')

# Render the final result.
plt.show()